# Import

In [3]:
import gzip
import pickle
import numpy as np
import random
from pathlib import Path

# Dataset path

In [4]:
TRAIN_PATH = "../dataset/train.pckl.gz"
TEST_PATH  = "../dataset/test.pckl.gzip"


# Load data

In [5]:

with gzip.open(TRAIN_PATH, "rb") as f:
    train_df = pickle.load(f)

with gzip.open(TEST_PATH, "rb") as f:
    test_df = pickle.load(f)

print(len(train_df), len(test_df))

24232 4242


# Build labeled ASE atoms list

In [12]:
def df_to_atoms(df):
    atoms_out = []
    
    for _, row in df.iterrows():
        a = row["ase_atoms"].copy()
        a.info["energy"] = float(row["energy_corrected"])
        a.arrays["forces"] = np.array(row["forces"])
        atoms_out.append(a)
        
    return atoms_out


train_atoms = df_to_atoms(train_df)
test_atoms  = df_to_atoms(test_df)

print(len(train_atoms), len(test_atoms))

24232 4242


# Sanity checks

In [22]:
a = test_atoms[0]

print("Energy:", a.info["energy"])
print("Forces shape:", a.arrays["forces"].shape)
print("Natoms:", len(a))
print("info keys:", a.info.keys())
print("arrays keys:", a.arrays.keys())

Energy: -381.796152188
Forces shape: (108, 3)
Natoms: 108
info keys: dict_keys(['energy'])
arrays keys: dict_keys(['numbers', 'positions', 'forces'])


# Export directly to extxyz

In [31]:
from ase.io import write
write("../dataset/train.extxyz", train_atoms, format="extxyz")
write("../dataset/test.extxyz", test_atoms, format="extxyz")

# Small debug subsets (still useful)

## Shuffle

In [32]:
import random

random.seed(42)

train_shuffled = train_atoms.copy()
test_shuffled  = test_atoms.copy()

random.shuffle(train_shuffled)
random.shuffle(test_shuffled)

## Write debug subsets

In [33]:
debug_train = train_shuffled[:200]
debug_test  = test_shuffled[:50]

write_extxyz("../dataset/train_debug.extxyz", debug_train)
write_extxyz("../dataset/test_debug.extxyz", debug_test )